In [1]:
import pandas as pd

In [2]:
# die ganzen Imports
from sklearn.model_selection import train_test_split

In [3]:
# train.csv einlesen
train_file_path = "train.csv"
f1_data = pd.read_csv(train_file_path)

In [4]:
# train.csv einlesen
train_file_path = "train.csv"
f1_data = pd.read_csv(train_file_path)

In [5]:
# y zuteilen --> PitNextLap
y = f1_data.PitNextLap

In [6]:
# feature erstellen und X zuteilen
features = [
    "Stint",
    "TyreLife",
    "Position",
    "Cumulative_Degradation",
    "RaceProgress",
    "Position_Change",
    "LapTime_Delta",
]

X = f1_data[features]

In [7]:
# in Validierung und Trainigsdaten aufteilen
train_X, val_X, train_y, val_y = train_test_split(X, y, random_state=1)

In [8]:
# Definiere das RandomForest Model
# rf_model = RandomForestRegressor(random_state=1)
# rf_model.fit(train_X, train_y)
# rf_val_predictions = rf_model.predict(val_X)
# rf_val_mae = mean_absolute_error(rf_val_predictions, val_y)

# print("Validation MAE for RF-Model: {:,.0f}".format(rf_val_mae))

In [9]:
# RandomForest Klassifizierer importieren
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score

In [10]:
# Definiere RandomForest Klassifizierung Model --> Klassifizerung und nicht Regression da kein € Wert getestet wird sondern ein Prozentwert zwischen 0 und 1
# habe im zweiten schritt N-estimators=300 gesetzt, Standardgemäß sind 100 Bäume in meinem RF, nun habe ich die Zahl auf 300 gesetzt
# mit min_samples_leaf=3 will ich verhindern dass runden auswendig gelernt werden
rf_model = RandomForestClassifier(n_estimators=200, min_samples_leaf=3, random_state=1)
rf_model.fit(train_X, train_y)

# predict_proba liefert Wahrscheinlichkeiten für beide Antwortmöglichkeiten 0 oder 1; [:, 1] schneidet uns die Wahrscheinlichkeit für einen echten Boxenstopp (1) heraus
rf_val_probabilities = rf_model.predict_proba(val_X)[:, 1]

rf_val_auc = roc_auc_score(val_y, rf_val_probabilities)

print("Validation ROC-AUC-Score für dein RF-Model: {:.4f}".format(rf_val_auc))

### Test wegen Overfitting ###
# Vorhersagen für die Trainingsdaten generieren
train_probs = rf_model.predict_proba(train_X)[:, 1]

# Score für die Trainingsdaten berechnen
train_auc = roc_auc_score(train_y, train_probs)

# Trainingsscore
print(f"Trainings-Score:     {train_auc:.4f}")

Validation ROC-AUC-Score für dein RF-Model: 0.9247
Trainings-Score:     0.9910


In [11]:
# test.csv einlesen
test_data_path = "test.csv"
test_data = pd.read_csv(test_data_path)

In [12]:
# test_X erstellen und es werden nur die Spalten genommen von dem feature
test_X = test_data[features]

In [13]:
# Vorhersage machen die wir hochladen
rf_model_on_full_data = RandomForestClassifier(random_state=1)

# auf allen Daten trainieren
rf_model_on_full_data.fit(X, y)

# Wahrscheinlichkeit für die Testdaten(X_test_ berechnen
test_preds = rf_model_on_full_data.predict_proba(test_X)[:, 1]

In [14]:
# Tabelle so aufbauen dass ich sie submitten kann
output = pd.DataFrame({"id": test_data.id, "PitNextLap": test_preds})

# Speichert das Ganze als submission.csv
output.to_csv("submission.csv", index=False)